In [ ]:
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
import joblib
df = pd.read_csv("../tas2_yontem1.csv",encoding="utf-8")
df['max_price'] = df['max_price'].fillna(df['max_price'].median())
df['min_price'] = df['min_price'].fillna(df['min_price'].median())
df['gender'] = df['gender'].fillna("NULL")
df['message'] = df['message'].fillna("NULL")
df['comment'] = df['comment'].fillna("NULL")
df['deviceType'] = df['deviceType'].fillna("NULL")
df['categories'] = df['categories'].fillna("NULL")
df['contents'] = df['contents'].fillna("NULL")



numerical_features = df.select_dtypes(include=['float64', 'int64']).columns

categorical_features = df.select_dtypes(include=['object']).columns

ss = StandardScaler()
ohe = OneHotEncoder(handle_unknown='ignore')
preprocessor = ColumnTransformer(
    transformers=[
        ('num', ss, numerical_features),
        ('cat', ohe, categorical_features)
    ]
)

for n_clusters in [2,3,5]:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('kmeans', kmeans)])
    pipeline.fit(df)
    df[f'cluster_{n_clusters}'] = pipeline.predict(df)
    joblib.dump(pipeline, f'kmeans_pipeline_{n_clusters}.pkl')
